List all the team members BITS ID ,Name along with % of contribution in this assignment:
1. 2024TM93056 - Mallidi Akhil Reddy - 100%
2. 2024TM93057 - Ashwarya Anupam - 100%
3. 2024TM93058 - Vaidya Rucha Sandeep - 100%
4. 2024TM93059 - Shivam Prabhakar - 100%
5. 2024TM93061 - J Rakesh - 100%

In [ ]:
# Catch-Up Game with Min-Max
# Instructions:
#  - When prompted enter n (e.g., 5 or 6).
#  - Choose game mode (1: Human vs AI, 2: AI vs AI).
#  - If Human vs AI you can choose to play P1 (first) or P2 (second).

import itertools
import functools
import math
import sys

# ---------- Utility: generate minimal subsets whose sum >= threshold ----------
def generate_minimal_subsets(remaining, threshold):
    """
    Return list of subsets (as tuples sorted ascending) of 'remaining' 
    that are minimal and whose sum >= threshold.
    Minimal means no proper subset has sum >= threshold.
    """
    remaining = sorted(remaining)
    results = []
    # We'll search by increasing size to find minimal ones early.
    for r in range(1, len(remaining)+1):
        for comb in itertools.combinations(remaining, r):
            s = sum(comb)
            if s >= threshold:
                # check minimality: every proper subset sum < threshold
                is_minimal = True
                for k in range(1, r):
                    for sub in itertools.combinations(comb, k):
                        if sum(sub) >= threshold:
                            is_minimal = False
                            break
                    if not is_minimal:
                        break
                if is_minimal:
                    results.append(tuple(comb))
        # We do not break here because there may be minimal combos of larger sizes
        # (e.g., if threshold is larger than any single element)
    return results

# ---------- Generate first-move options for P1: single elements ----------
def generate_first_moves(remaining):
    return [(x,) for x in remaining]

# ---------- Game state representation ----------
# We'll represent state as:
# (remaining_tuple_sorted, p1_total, p2_total, prev_turn_sum, player_to_move)
# player_to_move: 1 for P1, -1 for P2
# prev_turn_sum: the last turn-sum made by the other player (0 for start before any turns except after P1 initial it will be set)
#
# For Min-Max we'll return best score = (P1_total - P2_total) at terminal states

def state_key(remaining, p1_total, p2_total, prev_turn_sum, player):
    return (tuple(remaining), p1_total, p2_total, prev_turn_sum, player)

# ---------- Min-Max with memoization ----------
@functools.lru_cache(maxsize=None)
def minimax_cached(remaining_t, p1_total, p2_total, prev_turn_sum, player):
    # Convert tuple back to list for easier usage
    remaining = list(remaining_t)
    # Terminal?
    if not remaining:
        return p1_total - p2_total
    
    # Current player to move
    if player == 1:
        best = -10**9
    else:
        best = 10**9
    
    # Generate legal moves:
    if player == 1 and prev_turn_sum is None:
        # First move: P1 must choose exactly one number
        moves = generate_first_moves(remaining)
    else:
        threshold = prev_turn_sum if prev_turn_sum is not None else 0
        moves = generate_minimal_subsets(remaining, threshold)
        # If no move is available (shouldn't happen since threshold=0 only at start),
        # we allow picking any single remaining number (fallback).
        if not moves:
            moves = generate_first_moves(remaining)
    
    # For each move evaluate next states
    for move in moves:
        new_remaining = remaining.copy()
        for v in move:
            new_remaining.remove(v)
        # Update totals and prev_turn_sum for next player
        if player == 1:
            new_p1 = p1_total + sum(move)
            new_p2 = p2_total
            next_player = -1
        else:
            new_p1 = p1_total
            new_p2 = p2_total + sum(move)
            next_player = 1
        next_prev_turn_sum = sum(move)  # this player's turn sum becomes threshold for the opponent
        
        val = minimax_cached(tuple(sorted(new_remaining)), new_p1, new_p2, next_prev_turn_sum, next_player)
        if player == 1:
            if val > best:
                best = val
        else:
            if val < best:
                best = val
    return best

# ---------- Helper to get best move from Min-Max ----------
def best_move_minimax(remaining, p1_total, p2_total, prev_turn_sum, player):
    remaining_sorted = sorted(remaining)
    if player == 1 and prev_turn_sum is None:
        moves = generate_first_moves(remaining_sorted)
    else:
        threshold = prev_turn_sum if prev_turn_sum is not None else 0
        moves = generate_minimal_subsets(remaining_sorted, threshold)
        if not moves:
            moves = generate_first_moves(remaining_sorted)
    best_val = -10**9 if player == 1 else 10**9
    best_moves = []
    for move in moves:
        new_remaining = remaining_sorted.copy()
        for v in move:
            new_remaining.remove(v)
        if player == 1:
            new_p1 = p1_total + sum(move)
            new_p2 = p2_total
            next_player = -1
        else:
            new_p1 = p1_total
            new_p2 = p2_total + sum(move)
            next_player = 1
        next_prev_turn_sum = sum(move)
        val = minimax_cached(tuple(sorted(new_remaining)), new_p1, new_p2, next_prev_turn_sum, next_player)
        if player == 1:
            if val > best_val:
                best_val = val
                best_moves = [move]
            elif val == best_val:
                best_moves.append(move)
        else:
            if val < best_val:
                best_val = val
                best_moves = [move]
            elif val == best_val:
                best_moves.append(move)
    # Return any of the best moves (deterministic ordering)
    if not best_moves:
        return None
    return sorted(best_moves)[0]  # pick the lexicographically smallest best move for stable output

# ---------- Interactive game runner ----------
def run_game():
    print("=== Catch-Up Game ===")
    try:
        n = int(input("Enter n (highest number in set {1..n}), e.g. 5: ").strip())
    except Exception:
        print("Invalid input. Using n=5.")
        n = 5
    remaining = list(range(1, n+1))
    # Choose mode
    print("\nGame modes:\n 1 = Human vs AI\n 2 = AI vs AI (both play optimally)")
    mode = input("Choose mode [1 or 2] (default 1): ").strip()
    if mode not in ("1", "2"):
        mode = "1"
    human_is = None
    if mode == "1":
        # Ask whether human plays P1 or P2
        h = input("Do you want to play as P1 (first) or P2 (second)? Enter 1 or 2 (default 1): ").strip()
        if h == "2":
            human_is = -1
        else:
            human_is = 1
    
    p1_total = 0
    p2_total = 0
    prev_turn_sum = None  # for P1's first move special case
    player = 1  # 1 = P1's turn, -1 = P2's turn
    turn_no = 1
    print("\nStarting game. Initial numbers:", remaining)
    print("Rules summary: P1 first picks exactly one number. Afterwards each player's turn must pick numbers so that the sum of numbers chosen in that turn >= opponent's previous turn-sum. Player must stop as soon as they reach/exceed it (minimal picks).")
    print("We display each turn. When asked for human moves, valid moves are listed.\n")
    
    while remaining:
        print("-" * 60)
        print(f"Turn {turn_no}: {'P1' if player == 1 else 'P2'} to move.")
        print("Available numbers:", remaining)
        threshold_desc = "start (first move)" if (player == 1 and prev_turn_sum is None) else f"opponent's previous turn-sum = {prev_turn_sum}"
        print("Threshold:", threshold_desc)
        
        # Determine legal moves
        if player == 1 and prev_turn_sum is None:
            legal_moves = generate_first_moves(remaining)
        else:
            thr = prev_turn_sum if prev_turn_sum is not None else 0
            legal_moves = generate_minimal_subsets(remaining, thr)
            if not legal_moves:
                legal_moves = generate_first_moves(remaining)  # fallback
        
        # Present moves
        # For human, list them; for AI, choose via Min-Max
        if mode == "2" or (mode == "1" and ((player == 1 and human_is != 1) or (player == -1 and human_is != -1))):
            # AI move
            move = best_move_minimax(remaining, p1_total, p2_total, prev_turn_sum, player)
            print("AI chooses:", list(move))
        else:
            # Human move
            print("Legal moves (choose by index):")
            for idx, mv in enumerate(legal_moves):
                print(f"  {idx}: {list(mv)} (turn-sum = {sum(mv)})")
            chosen_idx = input(f"Enter move index (0..{len(legal_moves)-1}): ").strip()
            try:
                ci = int(chosen_idx)
                if ci < 0 or ci >= len(legal_moves):
                    raise ValueError()
            except Exception:
                print("Invalid index — choosing first legal move by default.")
                ci = 0
            move = legal_moves[ci]
            print("You chose:", list(move))
        
        # Apply the move
        for v in move:
            remaining.remove(v)
        if player == 1:
            p1_total += sum(move)
        else:
            p2_total += sum(move)
        print("Turn result -- move:", list(move), "turn-sum:", sum(move))
        print("Cumulative totals: P1 =", p1_total, ", P2 =", p2_total)
        # update for next player
        prev_turn_sum = sum(move)
        player = -player
        turn_no += 1
    
    # Game finished
    print("\n" + "="*60)
    print("Game over. All numbers chosen.")
    print("Final totals: P1 =", p1_total, ", P2 =", p2_total)
    if p1_total > p2_total:
        print("Winner: P1")
    elif p2_total > p1_total:
        print("Winner: P2")
    else:
        print("Result: Draw")
    print("="*60)

# ---------- Run ----------
if __name__ == "__main__":
    run_game()


=== Catch-Up Game ===


Enter n (highest number in set {1..n}), e.g. 5:  5



Game modes:
 1 = Human vs AI
 2 = AI vs AI (both play optimally)


Choose mode [1 or 2] (default 1):  1
Do you want to play as P1 (first) or P2 (second)? Enter 1 or 2 (default 1):  P1



Starting game. Initial numbers: [1, 2, 3, 4, 5]
Rules summary: P1 first picks exactly one number. Afterwards each player's turn must pick numbers so that the sum of numbers chosen in that turn >= opponent's previous turn-sum. Player must stop as soon as they reach/exceed it (minimal picks).
We display each turn. When asked for human moves, valid moves are listed.

------------------------------------------------------------
Turn 1: P1 to move.
Available numbers: [1, 2, 3, 4, 5]
Threshold: start (first move)
Legal moves (choose by index):
  0: [1] (turn-sum = 1)
  1: [2] (turn-sum = 2)
  2: [3] (turn-sum = 3)
  3: [4] (turn-sum = 4)
  4: [5] (turn-sum = 5)


Enter move index (0..4):  4


You chose: [5]
Turn result -- move: [5] turn-sum: 5
Cumulative totals: P1 = 5 , P2 = 0
------------------------------------------------------------
Turn 2: P2 to move.
Available numbers: [1, 2, 3, 4]
Threshold: opponent's previous turn-sum = 5
AI chooses: [3, 4]
Turn result -- move: [3, 4] turn-sum: 7
Cumulative totals: P1 = 5 , P2 = 7
------------------------------------------------------------
Turn 3: P1 to move.
Available numbers: [1, 2]
Threshold: opponent's previous turn-sum = 7
Legal moves (choose by index):
  0: [1] (turn-sum = 1)
  1: [2] (turn-sum = 2)


Enter move index (0..1):  1


You chose: [2]
Turn result -- move: [2] turn-sum: 2
Cumulative totals: P1 = 7 , P2 = 7
------------------------------------------------------------
Turn 4: P2 to move.
Available numbers: [1]
Threshold: opponent's previous turn-sum = 2
AI chooses: [1]
Turn result -- move: [1] turn-sum: 1
Cumulative totals: P1 = 7 , P2 = 8

Game over. All numbers chosen.
Final totals: P1 = 7 , P2 = 8
Winner: P2


In [ ]:
# Water Resource Prediction Decision Tree

def get_input(prompt, valid=None, vtype=float):
    raw = vtype(input(prompt).strip())
    return raw

def decision_tree():
    lake_distance = get_input("Enter distance from lake (km): ", lambda x: x >= 0, float)
    if lake_distance < 10:
        print("Recommended water source: Lake")
        return "Lake"
    river_distance = get_input("Enter distance from river (km): ", lambda x: x >= 0, float)
    if river_distance < 8:
        rainfall = get_input("Enter rainfall intensity (mm): ", lambda x: x >= 0, float)
        if rainfall >= 200:
            print("Recommended water source: Rain")
            return "Rain"
        else:
            print("Recommended water source: River")
            return "River"
    else:
        rainfall = get_input("Enter rainfall intensity (mm): ", lambda x: x >= 0, float)
        if rainfall >= 150:
            print("Recommended water source: Rain")
            return "Rain"
        else:
            aquifer = get_input("Is there a sandy aquifer (yes/no): ", ['yes', 'no'], str)
            if aquifer == "yes":
                beach_distance = get_input("Enter distance from beach (km): ", lambda x: x >= 0, float)
                if beach_distance < 5:
                    river_distance2 = get_input("Enter distance from river (km): ", lambda x: x >= 0, float)
                    if river_distance2 < 20:
                        print("Recommended water source: River")
                        return "River"
                    else:
                        print("Recommended water source: Rain")
                        return "Rain"
                else:
                    print("Recommended water source: Groundwater")
                    return "Groundwater"
            else:
                lake_distance2 = get_input("Enter distance from lake (km): ", lambda x: x >= 0, float)
                if lake_distance2 < 14:
                    print("Recommended water source: Lake")
                    return "Lake"
                else:
                    print("Recommended water source: Rain")
                    return "Rain"

if __name__ == "__main__":
    print("Welcome to the Water Resource Decision Tree System!")
    decision_tree()


Welcome to the Water Resource Decision Tree System!
Enter distance from lake (km): 10
Enter distance from river (km): 8
Enter rainfall intensity (mm): 130
Is there a sandy aquifer (yes/no): yes
Enter distance from beach (km): 2
Enter distance from river (km): 10
Recommended water source: River
